# Baseline cot_structured_sc + high_div_8 Eval

Run the base `Qwen/Qwen3-4B-Thinking-2507` model with the repo's `cot_structured_sc_only` run config. This uses `cot_structured_sc` plus `high_div_8` self-consistency and deliberately runs **without** a LoRA adapter.

## 1. GPU Check

In [ ]:
!nvidia-smi


## 2. Mount Drive and Locate Repo

In [ ]:
from pathlib import Path
import os
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

candidate_roots = []
if os.environ.get('REPO_ROOT'):
    candidate_roots.append(Path(os.environ['REPO_ROOT']).expanduser())
candidate_roots.extend([
    Path.cwd(),
    Path('/content/151B_SP26_Competition'),
    Path('/content/drive/MyDrive/qwen_math_comp/151B_SP26_Competition'),
])

REPO_ROOT = None
for root in candidate_roots:
    if (root / 'experiments/prompt_engineering/src/eval.py').exists():
        REPO_ROOT = root.resolve()
        break

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not find the repo root. Set os.environ["REPO_ROOT"] to the '
        '151B_SP26_Competition path and rerun this cell.'
    )

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ['REPO_ROOT'] = str(REPO_ROOT)

# Prevent accidental adapter use through shared.runner.ADAPTER_PATH fallback.
os.environ.pop('ADAPTER_PATH', None)

print('IN_COLAB =', IN_COLAB)
print('REPO_ROOT =', REPO_ROOT)
print('ADAPTER_PATH in env =', os.environ.get('ADAPTER_PATH'))


## 3. Install Dependencies

In [ ]:
%pip install -q -e ".[notebook]"


## 4. Baseline Run Config

Default target is the held-out public split, `data/test.jsonl`, with `max_tokens=16384`. Change `EVAL_DATA_PATH` to `data/public.jsonl` only if you want the full 1,126-row public-set sanity check.

In [ ]:
from pathlib import Path
import os

RESULTS_DIR = Path(os.environ.get(
    'RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else 'experiments/prompt_engineering/results',
))
EVAL_DATA_PATH = Path(os.environ.get('EVAL_DATA_PATH', 'data/test.jsonl'))
RUN_NAME = os.environ.get('RUN_NAME', 'base_cot_structured_sc_high_div_8_heldout_16384')
MAX_TOKENS = int(os.environ.get('MAX_TOKENS', '16384'))
RUNNER_MAX_MODEL_LEN = int(os.environ.get('RUNNER_MAX_MODEL_LEN', str(max(16384, MAX_TOKENS + 4096))))

# vLLM is the intended path. HF fallback with n=8 can be slow, so keep batches small.
RUNNER_MICRO_BATCH_SIZE = os.environ.get('RUNNER_MICRO_BATCH_SIZE', '2')
RUNNER_PARALLEL_SAMPLES = os.environ.get('RUNNER_PARALLEL_SAMPLES', '1')
WANDB_NAME = os.environ.get('WANDB_NAME', RUN_NAME)

if not EVAL_DATA_PATH.exists():
    raise FileNotFoundError(f'Eval data not found: {EVAL_DATA_PATH}')

os.environ['WANDB_NAME'] = WANDB_NAME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['RUNNER_MAX_MODEL_LEN'] = str(RUNNER_MAX_MODEL_LEN)
os.environ['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE
os.environ['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES
os.environ.pop('ADAPTER_PATH', None)

print('EVAL_DATA_PATH =', EVAL_DATA_PATH)
print('RESULTS_DIR =', RESULTS_DIR)
print('RUN_NAME =', RUN_NAME)
print('MAX_TOKENS =', MAX_TOKENS)
print('RUNNER_MAX_MODEL_LEN =', RUNNER_MAX_MODEL_LEN)
print('RUNNER_MICRO_BATCH_SIZE =', RUNNER_MICRO_BATCH_SIZE)
print('RUNNER_PARALLEL_SAMPLES =', RUNNER_PARALLEL_SAMPLES)
print('ADAPTER_PATH in env =', os.environ.get('ADAPTER_PATH'))


## 5. Verify Prompt and Regime

In [ ]:
from experiments.prompt_engineering.src.prompts import PROMPTS
from experiments.prompt_engineering.src.eval import _load_named_regime

prompt = PROMPTS['cot_structured_sc']
regime = _load_named_regime('high_div_8')

print('prompt id:', prompt.id)
print('prompt kind:', prompt.kind)
print('regime:', regime)
print('\nSystem prompt for free-response:\n')
print(prompt.system_free)
print('\nSystem prompt for MCQ:\n')
print(prompt.system_mcq)


## 6. Eval Helper

In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path


def build_eval_cmd(run_name: str, *, max_tokens: int = MAX_TOKENS, data_path: Path = EVAL_DATA_PATH, results_dir: Path = RESULTS_DIR, slice_indices=None) -> list[str]:
    cmd = [
        sys.executable,
        '-m', 'experiments.prompt_engineering.src.eval',
        'run=cot_structured_sc_only',
        'eval=full',
        f'eval.data_path={data_path}',
        f'eval.max_tokens={max_tokens}',
        'runner.engine=vllm',
        'runner.quant=bf16',
        'runner.adapter_path=null',
        f'results_dir={results_dir}',
        f'run_name={run_name}',
    ]
    if slice_indices is not None:
        joined = ','.join(str(int(i)) for i in slice_indices)
        cmd.insert(5, f'eval.slice_indices=[{joined}]')
    return cmd


def run_eval(run_name: str, *, max_tokens: int = MAX_TOKENS, data_path: Path = EVAL_DATA_PATH, results_dir: Path = RESULTS_DIR, slice_indices=None, micro_batch_size=None) -> None:
    env = os.environ.copy()
    env.pop('ADAPTER_PATH', None)
    env['WANDB_NAME'] = run_name
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['RUNNER_MAX_MODEL_LEN'] = str(max(16384, max_tokens + 4096))
    env['RUNNER_MICRO_BATCH_SIZE'] = str(micro_batch_size or RUNNER_MICRO_BATCH_SIZE)
    env['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES

    cmd = build_eval_cmd(
        run_name,
        max_tokens=max_tokens,
        data_path=data_path,
        results_dir=results_dir,
        slice_indices=slice_indices,
    )
    print('Running without adapter. ADAPTER_PATH in subprocess =', env.get('ADAPTER_PATH'))
    print('Command:')
    print(' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, env=env, check=True)


## 7. 5-Item Smoke Test

In [ ]:
run_eval(
    run_name=f'{RUN_NAME}_smoke5',
    slice_indices=[0, 1, 2, 3, 4],
    micro_batch_size=1,
)


## 8. Held-Out Baseline Eval

This evaluates every row in `EVAL_DATA_PATH`. With the default `data/test.jsonl`, that is the 226-row held-out split.

In [ ]:
run_eval(RUN_NAME)


## 9. Optional Full Public Eval

This includes the 900 training-split rows, so use it only to compare with older full-public logs.

In [ ]:
RUN_FULL_PUBLIC = False

if RUN_FULL_PUBLIC:
    run_eval(
        run_name='base_cot_structured_sc_high_div_8_full_public_16384',
        data_path=Path('data/public.jsonl'),
        max_tokens=MAX_TOKENS,
    )
else:
    print('Set RUN_FULL_PUBLIC = True to evaluate all 1,126 public rows.')
